# Подготовка датасета проектов для импорта в Django

1. Запусти ячейки сверху вниз до генерации промпта.
2. Скопируй вывод последней ячейки батча и вставь в LLM (один запрос на 50 проектов).
3. Вставь ответ модели в переменную `LLM_RESPONSE` в ячейке ниже и выполни её — получится `projects_enriched.csv` для `import_projects_dataset`.

Путь импорта в контейнере: `/app/src/data/projects_enriched.csv`.

In [1]:
from pathlib import Path
import json

import pandas as pd

# Пути (ноутбук в корне cursach)
ROOT = Path.cwd()
if not (ROOT / "data" / "requests_data.xlsx").exists():
    ROOT = ROOT.parent  # если запуск из подпапки

DATA_XLSX = ROOT / "data" / "requests_data.xlsx"
OUT_DIR = ROOT / "2025_hse_dsa" / "src" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 200
BATCH_SIZE = 50  # один большой промпт на столько проектов

In [2]:
df = pd.read_excel(DATA_XLSX)

col_id = "Номер вакансии/темы"
col_title = "Тема ВКР/КР"
col_email = "E-mail руководителя"

projects = (
    df[[col_id, col_title, col_email]]
    .dropna(subset=[col_id, col_title, col_email])
    .rename(
        columns={
            col_id: "project_id",
            col_title: "project_name",
            col_email: "professor_email",
        }
    )
)
projects["project_id"] = projects["project_id"].astype(int)
projects["professor_email"] = projects["professor_email"].str.strip().str.lower()

counts = (
    df[col_id]
    .value_counts()
    .rename_axis("project_id")
    .reset_index(name="applications_count")
)

top_projects = (
    projects.drop_duplicates(subset=["project_id"])
    .merge(counts, on="project_id", how="left")
    .sort_values(["applications_count", "project_id"], ascending=[False, True])
    .head(TOP_K)
    .reset_index(drop=True)
)

print(f"Строк в Excel: {len(df)}, уникальных проектов в top-{TOP_K}: {len(top_projects)}")
top_projects.head(3)

Строк в Excel: 29467, уникальных проектов в top-200: 200


,project_id,project_name,professor_email,applications_count
0,10512,Особенности трудоустройства специалистов покол...,mplakhotnik@hse.ru,15
1,1107,Разработка бизнес-стратегии,agbelyakov@hse.ru,14
2,3139,Трансмедийный сторителлинг во вселенной Гарри ...,dnazarova@hse.ru,14


In [3]:
TAG_VOCAB = [
    "Machine Learning",
    "Data Science",
    "Analytics",
    "Backend",
    "Frontend",
    "Web Development",
    "DevOps",
    "Product Management",
    "Finance",
    "Economics",
    "Marketing",
    "Strategy",
    "Research",
    "NLP",
    "Computer Vision",
    "Recommendation Systems",
    "Business Analysis",
    "Education",
    "Design",
    "Legal Tech",
]

KEYWORDS = {
    "машин": ["Machine Learning", "Data Science"],
    "данн": ["Data Science", "Analytics"],
    "аналит": ["Analytics", "Business Analysis"],
    "веб": ["Web Development", "Frontend", "Backend"],
    "сайт": ["Web Development", "Frontend"],
    "платформ": ["Backend", "Web Development"],
    "язык": ["NLP"],
    "модель": ["Machine Learning"],
    "рекоменд": ["Recommendation Systems", "Machine Learning"],
    "финанс": ["Finance", "Economics"],
    "эконом": ["Economics", "Analytics"],
    "маркет": ["Marketing", "Strategy"],
    "стратег": ["Strategy", "Product Management"],
    "исслед": ["Research"],
    "дизайн": ["Design"],
    "образован": ["Education"],
    "прав": ["Legal Tech", "Research"],
}


def bootstrap_tags(title: str) -> list[str]:
    title_norm = str(title or "").strip().lower()
    tags: list[str] = []
    for keyword, mapped in KEYWORDS.items():
        if keyword in title_norm:
            tags.extend(mapped)
    out: list[str] = []
    for t in tags:
        if t not in out:
            out.append(t)
    return out[:5] or ["Research"]


top_projects = top_projects.copy()
top_projects["bootstrap_tags"] = top_projects["project_name"].apply(bootstrap_tags)

In [4]:
def build_batch_prompt(batch_df: pd.DataFrame) -> str:
    vocab = ", ".join(f'"{t}"' for t in TAG_VOCAB)
    lines = [
        "Ты размечаешь университетские проектные темы для MVP рекомендательной системы.",
        "",
        "Верни ОДИН JSON-массив без markdown и без пояснений. Длина массива должна совпадать с числом проектов ниже.",
        "Каждый элемент массива строго такого вида:",
        '{"project_id": <int>, "description": "<2-4 предложения на русском>", "tags": ["tag1", "tag2", ...]}',
        "",
        "Правила:",
        "- description: реалистично, не выдумывай конкретные факты, которых нет в названии темы.",
        f"- tags: только из словаря: [{vocab}]",
        "- от 2 до 5 тегов на проект.",
        "- project_id должен точно совпадать с указанным в списке.",
        "",
        "Проекты:",
    ]
    for _, row in batch_df.iterrows():
        bt = ", ".join(row["bootstrap_tags"])
        lines.append(
            f"- project_id={int(row['project_id'])} | applications={int(row['applications_count'])} | "
            f"email={row['professor_email']} | title={row['project_name']!r} | bootstrap_tags=[{bt}]"
        )
    return "\n".join(lines)


def split_batches(df: pd.DataFrame, size: int) -> list[pd.DataFrame]:
    return [df.iloc[i : i + size].copy() for i in range(0, len(df), size)]


batches = split_batches(top_projects, BATCH_SIZE)
print(f"Батчей по {BATCH_SIZE}: {len(batches)}")

Батчей по 50: 4


### Один большой промпт (выбери номер батча)

Поменяй `BATCH_INDEX` с `0` по `len(batches)-1`, выполни ячейку, скопируй **весь** вывод в LLM.

In [15]:
BATCH_INDEX = 3  # 0, 1, 2, 3 для 200 проектов и BATCH_SIZE=50

prompt = build_batch_prompt(batches[BATCH_INDEX])
print(prompt)

Ты размечаешь университетские проектные темы для MVP рекомендательной системы.

Верни ОДИН JSON-массив без markdown и без пояснений. Длина массива должна совпадать с числом проектов ниже.
Каждый элемент массива строго такого вида:
{"project_id": <int>, "description": "<2-4 предложения на русском>", "tags": ["tag1", "tag2", ...]}

Правила:
- description: реалистично, не выдумывай конкретные факты, которых нет в названии темы.
- tags: только из словаря: ["Machine Learning", "Data Science", "Analytics", "Backend", "Frontend", "Web Development", "DevOps", "Product Management", "Finance", "Economics", "Marketing", "Strategy", "Research", "NLP", "Computer Vision", "Recommendation Systems", "Business Analysis", "Education", "Design", "Legal Tech"]
- от 2 до 5 тегов на проект.
- project_id должен точно совпадать с указанным в списке.

Проекты:
- project_id=9973 | applications=5 | email=mzhelezin@hse.ru | title='Применение методов машинного обучения для прогнозирования игровых действий в многоп

### После ответа LLM на батч

1. Вставь сырой ответ модели в `LLM_RESPONSE` (многострочная строка).
2. Убедись, что `BATCH_INDEX` совпадает с батчем, для которого копировал промпт.
3. Выполни ячейку — результат попадёт в `collected_labels`.
4. Повтори для всех батчей, затем собери CSV в последней ячейке.

In [ ]:
def parse_llm_json_array(text: str) -> list[dict]:
    text = text.strip()
    if text.startswith("```"):
        first_nl = text.find("\n")
        if first_nl != -1:
            text = text[first_nl + 1 :]
        if text.endswith("```"):
            text = text[: text.rfind("```")].rstrip()
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("В ответе нет JSON-массива [...]")
    return json.loads(text[start : end + 1])


# Накопитель по батчам (перезапусти ячейку с dict() если хочешь начать заново)
if "collected_labels" not in globals():
    collected_labels: dict[int, list[dict]] = {}

LLM_RESPONSE = r"""
Вставь сюда ответ модели целиком.
"""

BATCH_INDEX = 3  # тот же, что при генерации промпта

parsed = parse_llm_json_array(LLM_RESPONSE)
batch_df = batches[BATCH_INDEX]
expected_ids = set(batch_df["project_id"].astype(int))
got_ids = {int(x["project_id"]) for x in parsed}
if expected_ids != got_ids:
    missing = expected_ids - got_ids
    extra = got_ids - expected_ids
    raise ValueError(f"Несовпадение id: missing={missing}, extra={extra}")

collected_labels[BATCH_INDEX] = parsed
print(f"Батч {BATCH_INDEX} сохранён, всего батчей в collected_labels: {len(collected_labels)}")

Батч 3 сохранён, всего батчей в collected_labels: 4


### Сборка `projects_enriched.csv`

Запускай, когда `len(collected_labels) == len(batches)`.

In [17]:
if len(collected_labels) != len(batches):
    raise RuntimeError(
        f"Нужно {len(batches)} батчей, сейчас {len(collected_labels)}. Ключи: {sorted(collected_labels)}"
    )

rows_flat: list[dict] = []
for i in range(len(batches)):
    rows_flat.extend(collected_labels[i])

labels_df = pd.DataFrame(rows_flat)
labels_df["project_id"] = labels_df["project_id"].astype(int)

enriched = top_projects.merge(
    labels_df[["project_id", "description", "tags"]],
    on="project_id",
    how="left",
    validate="one_to_one",
)

def tags_to_csv_cell(tags) -> str:
    if isinstance(tags, str):
        return tags
    if isinstance(tags, list):
        return ", ".join(str(t) for t in tags)
    raise TypeError(tags)


enriched["tags"] = enriched["tags"].apply(tags_to_csv_cell)
enriched["description"] = enriched["description"].fillna("").astype(str).str.strip()

out_csv = OUT_DIR / "projects_enriched.csv"
review_csv = OUT_DIR / "projects_for_labeling.csv"

enriched[["project_id", "project_name", "professor_email", "description", "tags"]].to_csv(
    out_csv, index=False, encoding="utf-8"
)
top_projects.to_csv(review_csv, index=False, encoding="utf-8")

print(f"OK -> {out_csv}")
print(f"review -> {review_csv}")

OK -> /Users/user/Desktop/Учеба/cursach/2025_hse_dsa/src/data/projects_enriched.csv
review -> /Users/user/Desktop/Учеба/cursach/2025_hse_dsa/src/data/projects_for_labeling.csv
